# Spark transformations
## What do we mean by transformations?
![transformations](images/transformations.png)
- In Spark we read the data from a data source and create one of the two things. 
    - DataFrames
        - The DataFrame is the programmatic interface for your data
        - Transformation : Here the programmatic approach is implemented when it comes to transformations
    - Database table
        - The Database table is the sql interface for your data.
        - Transformation : Here the sql approach is implemented when it comes to transformations
- Both the database tables and the dataframes are the same but two different interfaces.
-  Transformation is nothing but:
    - Combining DataFrames
    - Aggregating and Summarizing 
    - Applying functions and built-in transformations
    - Using built-in and column-level functions
    - Creating and using UDFs
    - Creating column expressions
    - Referencing rows/columns
### Working with DataFrame rows:
- Spark dataframe is a dataset of rows.
- Each row in the dataFrame is a single record represented by an object of type row.
- Most of the time we do not directly work with the entire row.
- However there are three specific senarios where we have to directly work with the row object.
    - Manually creating rows and dataFrame.
    - Collecting DataFrame rows to the driver.
    - Work with an individual row in spark transformations.
#### Example 1: 
- Here in this example you will see where creating a dataFrame manually on the fly helps in unit testing of the functions and method that you create
- Why creating dataFrame helps manually instead of importing sample data from a csv file?
    - Everytime we have to test a function or a method its not possible to read the csv file and bring in some sample data to create a dataFrame because it will make testing the application significantly slower due to un-necessary I/O overhead.
- In this example I have written a function that converts dataType from string to date type given the column name in a dataFrame
- I have also written a test case using python's built in unit test tool to simulate real world senario to the best of my ability.
#### Here is the final code 
#### dataframe_transformations.py
```python
from pyspark.sql.functions import to_date
class DataFrameTransformations:
    def __init__(self,spark):
        self.spark_object = spark
    def count_by_country(self,spark_df):
        intermediate_result_df = (
                spark_df
                .where("CallType is not null")
                .select("CallType","Zipcode")
                .groupby("CallType","Zipcode")
            )
        row_count = intermediate_result_df.count()
        result_df = row_count.orderBy("count",ascending=False)
        return result_df
    
    """This methods onverts the column with dates in string datatype to date datatype"""
    def convert_to_date_type(self, spark_df, date_format, col_name):
        """Converts a string column to date using the given format."""
        return spark_df.withColumn(col_name, to_date(col_name, date_format))
```
#### unit_test.py
```python
import unittest
import os 
import sys
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(os.path.dirname(CURRENT_DIR))  # one level higher
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from pyspark.sql import SparkSession, Row
from SparkDFTransformations.transformations.dataframe_transformations import DataFrameTransformations
from datetime import date


class TestDataFrameTransformations(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.spark = (
            SparkSession
            .builder
            .appName("PySparkUnitTest")
            .master("local[2]")
            .getOrCreate()
        )
        cls.transformer = DataFrameTransformations(cls.spark)

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()

    def test_convert_to_date_type(self):
        # Sample test DataFrame
        data = [
            Row(id="1", EventDate="3/11/2025"),
            Row(id="2", EventDate="4/11/2025")
        ]

        schema = "id STRING, EventDate STRING"
        df = self.spark.createDataFrame(data, schema)

        # Apply transformation
        result_df = self.transformer.convert_to_date_type(df, "d/M/yyyy", "EventDate")

        # Collect and check types
        result = result_df.collect()

        # Assert that EventDate is converted to a Python date object
        self.assertIsInstance(result[0]['EventDate'], date)
        self.assertEqual(result[0]['EventDate'], date(2025, 11, 3))

        print("✅ convert_to_date_type() test passed!")


if __name__ == '__main__':
    unittest.main()
```
#### Example 2: Ingest unstructured data from an apache.log file and perform dataFrame transformations on it.
- Here in this example I am going to simulate how to handle unstructured data in pyspark using a log_file from a server
- The data file is an apache webserver log file.
- We do have some pattern in this log file but this file is not even a semi-structure data file it is just a log dump
- So If I try to read this file into a dataFrame all I am going to get is a single row of strings
- I won't be getting columns in the dataFrame because the log file is an unstructured data file.
- In this case I won't be able to use many of the higher level transformation such as aggregation and grouping.
- We need to find a way to extract some well defined field from the data.
    - The way that I chose to extract data from the apache server log file is to use regex
    - I used this regex ```log_regex = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'``` to collect these information from the log file : 
    ``` bash
        IP
        client
        datetime
        cmd
        request
        protocol
        status
        bytes
        referrer
        userAgent
    ```
    - Example : 
    ```python
    log_regex = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'
    file_df = (
        self.spark_object
        .read
        .format("text")
        .load(file_dir)
    )
    spark_df = file_df.select(regexp_extract())
    ```
    - **Explaiantion :**
        - ```regexp_extract()``` this functionn takes in 3 arguments
            - The first argument is the source string or the field name
            - The second argument is the regular expression
                - This regular expression will extract all 11 fields
            - The third field is the index no of the the field to be selected and extract from the log text
        - Your final code should look something like this :
        ```python
        def import_data_text(self,file_dir,unstructured:bool=False):
        try:
            # This regex is used to extract these data from the log text file
            """
            IP
            client
            datetime
            cmd
            request
            protocol
            status
            bytes
            referrer
            userAgent
            """
            log_regex = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'
            file_df = (
                self.spark_object
                .read
                .format("text")
                .load(file_dir)
            )
            spark_df = file_df.select(
                            regexp_extract('value',log_regex,1).alias("ip"),
                            regexp_extract('value',log_regex,4).alias("date"),
                            regexp_extract('value',log_regex,6).alias("request"),
                            regexp_extract('value',log_regex,10).alias("referrer"),
                        )
            self.log_df_metrics(spark_df=spark_df,file_dir=file_dir)
            return spark_df
        except Exception as e:
            self.logger.error(str(e))
            raise
        ```
- Now I want to group the rows based on referrer column also I want it to be grouped based on the website's domain names instead of the entire url 
    - example : 
    ```bash
    http://semicomplete.com/presentations/logstash-monitorama-2013/
    ```
    to this 
    ```
    http://www.semicomplete.com 
    ```
    - Here is the final code : 
    ```python
    def groupby_referrer(self,spark_df, col_name):
        try:
            if col_name == "referrer":
                result_df = (
                        spark_df
                        .withColumn(col_name,F.substring_index(F.col(col_name),"/",3))
                        .groupBy(col_name)
                        .count()
                    )
            else:
                result_df = (
                    spark_df
                    .groupBy(col_name)
                    .count()
                )
            self.log_df_metrics(result_df,operation_name="groupby_referrer")
            return result_df
        except Exception as e:
            self.logger(str(e))
            raise
    ```
    - Explaiantion : 
        - ```F.col(col_name)``` : 
            - This creates a Spark Column object referring to the given column name — e.g. "referrer".
        - ```F.substring_index(F.col(col_name), "/", 3)```
            - The function substring_index is a string function in PySpark that:
                - Returns the substring from the start of the string up to the n-th occurrence of a delimiter.
                - Remember if you enter ```3``` then only first two substrings in between the delimiter will be considered 
                - str → The column or string you’re operating on.
                - delimiter → The character(s) to split by ("/" here).
                - count → How many parts (delimiters) to include.
                - ```/``` means split by /:
                ```bash
                ['http:', '', 'semicomplete.com', 'presentations', 'logstash-monitorama-2013', '']
                ```
                - ```3``` count = 3:
                ```python
                'http://semicomplete.com'
                ```
        - ```withColumn()```
            - This replaces the referrer column with only its domain-level substring
            - When you later do .groupBy("referrer").count(), Spark groups all hits from the same domain together.
- **FINAL CODE** : You will get the final code in this github repo : https://github.com/aryan68125/Deep-learning-prerequisite/tree/master/pyspark/SparkTransformations/SparkDFTransformations

### Working with DataFrame columns: 
- **What is a column and how to reference it?**
    - Spark dataFrame columns are the objects of type column.
    - These column objects do not make any sense outside the context of the dataFrame and you cannot manipulate them independently.
    - Columns are always use within a spark dataFrame transformations.
    - There are two ways to refer to columns in a dataFrame transformation
        - Column String
            - Column string is the simplest method to access the column
            - Example : 
            ```python
            def select_col(self,spark_df, saprk_df_name:str = "",col_list:list = []):
            try:
                if len(col_list):
                    return spark_df.select(*col_list)
                else:
                    self.logger.error(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
                    raise ValueError(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
            except Exception as e:
                self.logger.error(str(e))
                raise
            ```
        - Column object
- **How to create column expressions?**
    - In Spark you can create a column expression using two types.
        - String expressions or SQL expressions
        ```python
        # defined in transformation.py
        def select_col(self,spark_df, saprk_df_name:str = "",col_list:list = [], expr_str:str=""):
        try:
            if len(col_list) and not expr_str:
                return spark_df.select(*col_list)
            elif len(col_list) and expr_str:
                return spark_df.select(*col_list, F.expr(expr_str))
            else:
                self.logger.error(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
                raise ValueError(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
        except Exception as e:
            self.logger.error(str(e))
            raise

        # usage in main.py file
        spark_df_selected_col = df_t.select_col(spark_df=spark_df_csv,saprk_df_name="spark_df_csv",col_list=["FL_DATE","ORIGIN_CITY_NAME", "DEST_CITY_NAME", "DISTANCE"],expr_str="DISTANCE * 1.609344 as DISTANCE_KM")
        ```
        - Column Object expressions
        **main.py**
        ```python
        from pyspark.sql import SparkSession
        # import related to logging
        from lib.logger import Log4j, LogSparkDataframe
        # import related to custom spark configurations
        from lib.utils import get_spark_app_config
        # imports related to exporting dataframe
        from lib.write_df import ExportSparkDataFrame
        # import writing sparkdf to tables related stuff
        from lib.load_df_data_into_table import LoadSparkDFIntoTable
        # logging related imports 
        import os

        # Imports related to ingest data
        from lib.ingest_data import IngestData
        # Transform data
        from transformations.dataframe_transformations import DataFrameTransformations

        # imports related to cleanup when the main_app.py is re-run
        from lib.clean_up_file_system import CleanupAppFileSystemOnReRun

        if __name__ == "__main__":
            # logging related logic
            # Get the current project's directory
            project_dir = os.path.dirname(os.path.abspath(__file__))
            # cleanup loggic on main_app.py re-run
            # initialize the cleanup class
            cleanup = CleanupAppFileSystemOnReRun(project_dir)
            cleanup.execute_cleanup(clean_logs=True)

            # Get the Log4j.properties file directory
            log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
            # Save the directory where the generated log files must reside
            log_dir = os.path.join(project_dir, "log4j_properties", "logs")
            # Create the directory where the log files must be kept if not present
            os.makedirs(log_dir, exist_ok=True)

            conf = get_spark_app_config()
            spark = (
                SparkSession
                .builder
                .config(conf=conf)
                .config("spark.driver.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .config("spark.executor.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.0.1")
                .enableHiveSupport()
                .getOrCreate()
            )

            # initialize logger class 
            logger = Log4j(spark)

            # initialize the spark dataframe logger 
            sp_df_logger = LogSparkDataframe(spark)

            # logging some debug related stuff 
            logger.debug(f"log4j.properties file dir = {log4j_config_path}")
            logger.debug(f"log files dir = {log_dir}")
            logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
            
            logger.info("Reading the data from the directory")
            dataset_dir = os.path.join(project_dir,"dataset")

            # Simulating unstructured data ingested from apache server logs STARTS
            """Import data from a log file (unstructured data) STARTS"""
            # import data from a text file 
            file_name = conf.get("file_name_text")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            ingest_data = IngestData(spark)
            spark_df_text = ingest_data.import_data_text(file_dir=file_dir,unstructured=True)
            # log spark_df_parquet dataframe
            sp_df_logger.log_df(spark_df=spark_df_text,spark_df_name="spark_df_text")
            """Import data from a log file (unstructured data) ENDS"""

            """Data Transformation STARTS"""
            df_t = DataFrameTransformations(spark)

            # conver the string dataType datetime to timestamp datatype datetime
            spark_df_text = df_t.convert_str_to_timestamp_type(spark_df=spark_df_text,col_name="date",spark_df_name="spark_df_text")
            sp_df_logger.log_df(spark_df=spark_df_text,spark_df_name="spark_df_text")

            # groupBy() rows based on referrer column
            spark_df_text = df_t.groupby_referrer(spark_df=spark_df_text, col_name="referrer")
            sp_df_logger.log_df(spark_df=spark_df_text,spark_df_name="spark_df_text")
            """Data Transformation ENDS"""
            # Simulating unstructured data ingested from apache server logs ENDS

            # Working with dataFrame columns STARTS
            """Import data from a paraquet file STARTS"""
            # import data from a text file 
            file_name = conf.get("file_name_csv")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            ingest_data = IngestData(spark)
            spark_df_csv = ingest_data.import_data_csv(file_dir=file_dir)
            # log spark_df_csv dataframe
            sp_df_logger.log_df(spark_df=spark_df_csv,spark_df_name="spark_df_csv")
            """Import data from a paraquet file ENDS"""

            """Transformation STARTS"""
            df_t = DataFrameTransformations(spark)

            # select column from a dataFrame
            spark_df_bool = df_t.select_col(
                spark_df=spark_df_csv,
                saprk_df_name="spark_df_csv",
                col_list=["FL_DATE", "CANCELLED", "DISTANCE"],
                convert_to_bool_col_name="CANCELLED"
                )
            # log spark_df_csv dataframe
            sp_df_logger.log_df(spark_df=spark_df_bool,spark_df_name="spark_df_selected_col")
            sp_df_logger.log_df_metrics(spark_df=spark_df_bool, spark_df_name="spark_df_selected_col")

            # Convert miles to km
            spark_df_selected_col = df_t.select_col(spark_df=spark_df_csv,saprk_df_name="spark_df_csv",col_list=["FL_DATE","ORIGIN_CITY_NAME", "DEST_CITY_NAME", "DISTANCE"],expr_str="DISTANCE * 1.609344 as DISTANCE_KM")
            # log spark_df_csv dataframe
            sp_df_logger.log_df(spark_df=spark_df_selected_col,spark_df_name="spark_df_selected_col")
            sp_df_logger.log_df_metrics(spark_df=spark_df_selected_col, spark_df_name="spark_df_selected_col")
            
            # Convert CANCELLED column data from integer 0 and 1 to True and False
            spark_df_cancelled_col_bool = df_t.select_col(spark_df=spark_df_csv,saprk_df_name="spark_df_csv",col_list=["FL_DATE","ORIGIN_CITY_NAME", "DEST_CITY_NAME", "DISTANCE","CANCELLED"],convert_to_bool_col_name="CANCELLED")
            # log spark_df_csv dataframe
            sp_df_logger.log_df(spark_df=spark_df_cancelled_col_bool,spark_df_name="spark_df_cancelled_col_bool")
            sp_df_logger.log_df_metrics(spark_df=spark_df_cancelled_col_bool, spark_df_name="spark_df_cancelled_col_bool")
            """Transformation ENDS"""
            # Working with dataFrame columns ENDS
            
            # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
            # input("Please enter")
            spark.stop()
        ```
        **transformation.py**
        ```python
        import os 
        import sys
        CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
        print(f"CURRENT_DIR >> {CURRENT_DIR}")
        PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
        print(f"PROJECT_ROOT >> {PROJECT_ROOT}")
        if PROJECT_ROOT not in sys.path:
            sys.path.insert(0, PROJECT_ROOT)
        print(f"printing sys path of python >>>")
        for p in sys.path[:5]:
            print("  ", p)

        # import transformation related stuff
        from pyspark.sql import functions as F
        from pyspark.sql.types import BooleanType
        from pyspark.sql.functions import udf
        # import logging related stuff
        from lib.logger import Log4j
        from lib.app_monitor import GetDataFrameMemory

        class DataFrameTransformations:
            def __init__(self,spark):
                self.spark_object = spark
                self.logger = Log4j(spark)
                self.metrics = GetDataFrameMemory(spark)


            """This methods onverts the column with dates in string datatype to date datatype"""
            def convert_str_to_timestamp_type(self, spark_df, col_name,spark_df_name):
                try:
                    self.logger.debug(f"converting date columns from string datatype to timestamp datatype in dataFrame {spark_df_name}")
                    # Apply Spark’s to_timestamp() with the exact format
                    if spark_df_name == "spark_df_text" and col_name=="date":
                        parsed_col = F.to_timestamp(F.col(col_name), "dd/MMM/yyyy:HH:mm:ss Z")
                    else:
                        self.logger.error(f"You need to implement the rules to related to dataType conversion to date type from string type")
                        return None

                    result_df = spark_df.withColumn(col_name, parsed_col)
                    self.log_df_metrics(result_df,operation_name="convert_str_to_timestamp_type")
                    return result_df
                except Exception as e:
                    self.logger.error(str(e))
                    raise
            
            """this method groups the data based on referrer"""
            def groupby_referrer(self,spark_df, col_name):
                try:
                    if col_name == "referrer":
                        result_df = (
                                spark_df
                                .where(f"trim({col_name}) != '-'")
                                .withColumn(col_name,F.substring_index(F.col(col_name),"/",3))
                                .groupBy(col_name)
                                .count()
                            )
                    else:
                        result_df = (
                            spark_df
                            .where(f"trim({col_name}) != '-'")
                            .groupBy(col_name)
                            .count()
                        )
                    self.log_df_metrics(result_df,operation_name="groupby_referrer")
                    return result_df
                except Exception as e:
                    self.logger.error(str(e))
                    raise

            """This method will select the columns based on column_name"""
            def select_col(self,spark_df, saprk_df_name:str = "",col_list:list = [], expr_str:str="",convert_to_bool_col_name:str=""):
                try:
                    if len(col_list) and not expr_str and not convert_to_bool_col_name:
                        return spark_df.select(*col_list)
                    elif len(col_list) and expr_str and not convert_to_bool_col_name:
                        return spark_df.select(*col_list, F.expr(expr_str))
                    elif len(col_list) and not expr_str and convert_to_bool_col_name:
                        bool_udf = udf(self.bool_parser, BooleanType())
                        result_df = spark_df.withColumn(
                            convert_to_bool_col_name,
                            bool_udf(F.col(convert_to_bool_col_name))
                        )
                        return result_df.select(*col_list, convert_to_bool_col_name)
                    else:
                        self.logger.error(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
                        raise ValueError(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
                except Exception as e:
                    self.logger.error(str(e))
                    raise

            # utility methods
            @staticmethod
            def bool_parser(bool_val):
                if bool_val in (1, "1", True):
                    return True
                elif bool_val in (0, "0", False):
                    return False
                else:
                    return None  # handle unexpected cases gracefully

            def log_df_metrics(self,spark_df,operation_name):
                self.logger.info(f"{operation_name} :: The memory taken by the spark dataFrame is = {self.metrics.get_mem_usage(spark_df).get("mem")} MB")
                schema_str = spark_df._jdf.schema().treeString()
                self.logger.debug(f"Spark DataFrame Schema (expanded): {schema_str}")
        ```
        - Explaination:
        - Focus in select_col method 
        ```python
            bool_udf = udf(self.bool_parser, BooleanType())
                            result_df = spark_df.withColumn(
                                convert_to_bool_col_name,
                                bool_udf(F.col(convert_to_bool_col_name))
                            )
        ```
        - This class method (select_col) converts a column containing numeric or string flags like "1", "0" into Boolean (True/False)
        - But Spark doesn’t have built-in logic to handle "1"/"0" or string booleans.
        - So I define our own Python function (bool_parser) and wrap it with Spark’s udf() mechanism.
        - ```bool_udf = udf(self.bool_parser, BooleanType())```
            - ```self.bool_parser``` → This is my Python function where it checks if the value is 1 and replaces it with the truth value and replaces 0th value with the false value
            - ```udf(self.bool_parser, BooleanType())``` wraps that Python function into a Spark UDF that can be applied on DataFrame columns distributedly.
            - BooleanType() tells Spark that the function will return boolean values (True/False).
        - ```spark_df.withColumn()``` 
            - It is used for column transformations in PySpark.
            - Creates a new DataFrame by adding, replacing, or transforming a column.
            - In this case it creates a new DataFrame with the same columns as spark_df, except it replaces or adds one column.
            - ```convert_to_bool_col_name``` the column name you want to transform (e.g., "flag").
            - ```bool_udf(F.col(convert_to_bool_col_name))``` Applies my UDF to every value in that column
            - Essentially, this line tells Spark: For each row, take the column flag, run bool_parser() on it, and store the result back in flag
        - ```return result_df.select(*col_list, convert_to_bool_col_name)``` This simply selects the original columns I asked for (col_list) plus the converted boolean column.
## Some miscelleneous transformations
- In this I am going to create a dataFrame and do some transformation opertions on it 
### Final code : 

### Explaiantion:
- ```generated_df = self.spark.createDataFrame(data_list).toDF("name","day","month","year")```
    - When I call createDataFrame(data_list), Spark will automatically
        - Convert it into a distributed DataFrame
        - Infer the schema (column names as _1, _2, _3, … by default)
        - Infer data types (string, integer, etc.)
- ```.toDF("name", "day", "month", "year")```
    - The .toDF() method renames the columns of the DataFrame.
        - By default, Spark names columns as _1, _2, _3, etc.
        ```bash
        +-----+---+--------+----+
        |  _1 | _2|   _3   | _4 |
        +-----+---+--------+----+
        |Alice| 12|January |2025|
        |  Bob|  3|March   |2024|
        +-----+---+--------+----+
        ```
        - .toDF() allows me to assign meaningful column names.
        ```bash
        +-----+---+--------+----+
        | name|day| month  |year|
        +-----+---+--------+----+
        |Alice| 12|January |2025|
        |  Bob|  3|March   |2024|
        +-----+---+--------+----+
        ```
- Now, the variable generated_df holds a PySpark DataFrame with:
    - Properly named columns
    - Properly inferred datatypes (Not so fast this dataFrame is still not done yet it's half baked you will see what I mean soon enough)
    - Ready for transformations, filters, joins, etc.
- Problems with this dataFrame
    - dates are in string dataType
    ```bash
    |-- name: string (nullable = true)
    |-- day: string (nullable = true)
    |-- month: string (nullable = true)
    |-- year: string (nullable = true)
    ```
    - inconsitent data in the year column, some year data is like this 2002 some year are like this 81 this causes issue in running alytical scripts on the dataFrame and we need to fix it asap 
    - there isn't a way to uniquely identify a row 
    ```bash
             name  day  month  year
           Rollex   28      1  2002
        Ballistic   23      5    81
          Shotgun   12     12     6
        Artillery    7      8    63
        Ballistic   23      5    81
    ```
- First I will be repartition the dataframe
    - ```python
        class GenerateDataFrame:
        def __init__(self, spark):
            self.spark = spark
            self.logger = Log4j(spark)
            self.sp_df_logger = LogSparkDataframe(spark)
            self.app_metrics = GetDataFrameMemory(spark)

        def generate_dataframe(self, data_list : List[Tuple[Any, ...]] = None):
            self.logger.debug(f"checking for the supplied data_list: ")
            if not data_list:
                self.logger.error("DataList required to generate a dataFrame!")
                raise ValueError(f"DataList required to generate a dataFrame!")
            self.logger.debug(f"supplied data_list found : {data_list}")

            # check if the spark session master is set to local if yes then implement repartition if not then don't
            if self.spark.sparkContext.master == "local[3]":
                generated_df = (
                    self.spark
                    .createDataFrame(data_list)
                    .toDF("name","day","month","year")
                    .repartition(3)
                )
            else:
                generated_df = (
                    self.spark
                    .createDataFrame(data_list)
                    .toDF("name","day","month","year")
                )
            self.app_metrics.get_mem_usage(generated_df)
            self.sp_df_logger.log_df_metrics(spark_df=generated_df,spark_df_name="generated_df")
            return generated_df
        ```
    - ```.repartition(3)``` 
        - Spark divides data into partitions, which are chunks of data processed in parallel.
        - repartition(3) tells Spark to create exactly 3 partitions of my DataFrame.
        - Each partition can be processed by one executor core in parallel.
        - Why only in local[3] mode:
            - local[3] means my Spark job is running locally with 3 CPU cores.
            - By calling .repartition(3), I am explicitly matching the number of partitions to the number of cores → achieving optimal parallelism.
        - Why not in YARN or Cluster:
            - When Spark runs on a real cluster (YARN, Kubernetes, etc.), partitioning is automatically handled by Spark’s cluster scheduler.
            - Forcing .repartition(3) in a large cluster would severely limit parallelism — only 3 partitions across many executors = underutilization.
        - The reason we are doing repartition when running the spark application locally in order to simulate the cluster in the production environment.
- Second I will be adding a unique indentifier to indentify the rows in the dataFrame uniquely.
    - ```python
        def create_unique_identifier(self,spark_df):
            try:
                spark_df = spark_df.withColumn("id",F.monotonically_increasing_id())
                return spark_df
            except Exception as e:
                self.logger.error(str(e))
                raise
        ```
        - The reason I am doing it because right now as you can see I don't have a way to uniquely identify a row and this becomes a problem when I want to get data from a particular row
        ```bash
             name  day  month  year
           Rollex   28      1  2002
        Ballistic   23      5    81
          Shotgun   12     12     6
        Artillery    7      8    63
        Ballistic   23      5    81
        ```
        - ```F.monotonically_increasing_id()```
            - Adds a new column called "id" that gives each row a unique numeric identifier.
            - Behavior:
                - Generates unique IDs across all partitions, but not sequential.
                - The IDs are increasing within a partition, but not across the entire dataset.
                - Example : 
                    - ```bash
                        | partition | row | generated ID |
                        | --------- | --- | ------------ |
                        | 0         | 1   | 0            |
                        | 0         | 2   | 1            |
                        | 1         | 1   | 8589934592   |
                        | 1         | 2   | 8589934593   |
                        ```
                    - Notice how the IDs “jump” by a large number (here, 2^33) between partitions.
                    - That’s because Spark uses the partition index in the upper bits of the ID to guarantee uniqueness in distributed mode.
                    - Key Notes:
                        - IDs are guaranteed unique but not continuous.
                        - Not suitable if you want strictly sequential row numbers.
                        - Works well for distributed datasets where you just need a unique identifier for each record.
                        - If you ever need a strictly sequential ID (like a database primary key), you’d use zipWithIndex() on an RDD or use a window function with row_number() — but those require extra shuffling.
- Third I will process the date's year column
    - ```bash
               name day month year          id
             Rollex  28     1 2002           0
          Ballistic  23     5   81           1
            Shotgun  12    12    6  8589934592
          Ballistic  23     5   81 17179869184
          Artillery   7     8   63 17179869185
      ```
    - Notice how the some of the year values are in double digits and single digits it is not a 4 digit number.
    - I want to address this data discrepency and make every value in the year column a 4 digit number
    - For this I am going to create a logic with a few assumtion
        - The value below 25 must be after the year 2000
        - The value below 100 must be in the year 1900
        - If the above two assumption fails then accept the value as it is.
    - NOTICE : This approach is not perfect you are responsible for developing the logic based on your needs and your own dataset using hit and trial method.
    - ```python
        def process_date_col_year(self,spark_df,col_name : str=None):
            try:
                if not col_name or col_name == "":
                    self.logger.error("col_name cannot be empty or None!")
                    raise ValueError("col_name cannot be empty or None!")
                spark_df = spark_df.withColumn(col_name,F.expr("""
                case when year < 25 then year + 2000 
                when year < 100 then year + 1900
                else year
                end
                """))
                return spark_df
            except Exception as e:
                self.logger.error(str(e))
                raise
        ```
        - ```python
            F.expr("""
                    case when year < 25 then year + 2000 
                    when year < 100 then year + 1900
                    else year
                    end
                    """)
            ```
        - Here I am using switch case sql logic to process the values in the year column
        - Final output :
        ```bash
               name day month  year          id
             Rollex  28     1  2002           0
          Ballistic  23     5  1981           1
            Shotgun  12    12  2006  8589934592
          Ballistic  23     5  1981 17179869184
          Artillery   7     8  1963 17179869185
        ```
- Fourth I will cast the field into its appropriate dataTypes in the dataFrame
    - NOTICE : I did not face this issue personally when implementing this application I am using pyspark 4.0.1 with jav 17 but this error may occur depending on the spark version also I am printing the output in the logs instead of the terminal. This may also be a contributing factor for the problem to not occur. Additional investigation need to be done but for now this will have to do. 
    - This is required because sometimes what can happen is 
        - The year column after the processing may return a values like this
        ```bash
               name day month  year            id
             Rollex  28     1  2002.0           0
          Ballistic  23     5  1981.0           1
            Shotgun  12    12  2006.0  8589934592
          Ballistic  23     5  1981.0 17179869184
          Artillery   7     8  1963.0 17179869185
        ```
        - This happens because 
            - Incorrect dataType
            - Automatic type promotion
        - The year field in the dataFrame is a string however we performed an arithematic operation in the year field.
        - So the spark sql engine automatically promoted it into decimal. 
        - After completing the arithematic operation it is again demoted back to string
        - This is the reason you may get the dataFrame as shown above after processing the year column
    - Solution : 
        - Cast each and every field of your dataFrame into its appropriate datatype
        - There are two ways you can do that:
            - Inline CAST : This is to make sure that spark does not automatically promote or demote your field type. Instead I need to write code to push it to an appropriate type.
            ```python
            def process_date_col_year(self,spark_df,col_name : str=None):
            try:
                if not col_name or col_name == "":
                    self.logger.error("col_name cannot be empty or None!")
                    raise ValueError("col_name cannot be empty or None!")
                spark_df = spark_df.withColumn(col_name,F.expr("""
                case when year < 25 then cast(year as int) + 2000 
                when year < 100 then (year as int) + 1900
                else year
                end
                """))
                return spark_df
            except Exception as e:
                self.logger.error(str(e))
                raise
            ```
            - ```cast(year as int)``` Now this will cast the values in the year column to an integer dataType.
            - Now the spark sql engine will not promote it and cause problem for us
            - Change the schema : This approach is pretty stright forward define a schema and then insert the data into the dataFrame so that we can avoid this pitfall
- Fourth I will add a new column combining values in the columns day, month and year 
    - ```python
            def process_date_col_year(self,spark_df,col_name : str=None, combine_date : bool = False):
                try:
                    if not col_name or col_name == "":
                        self.logger.error("col_name cannot be empty or None!")
                        raise ValueError("col_name cannot be empty or None!")
                    spark_df = spark_df.withColumn(col_name,F.expr("""
                    case when year < 25 then cast(year as int) + 2000 
                    when year < 100 then cast(year as int) + 1900
                    else year
                    end
                    """))

                    if combine_date:
                        spark_df = spark_df.withColumn("dob",F.expr("""
                            to_date(concat(day,'/',month,'/',year),'d/M/y')
                        """))
                    return spark_df
                except Exception as e:
                    self.logger.error(str(e))
                    raise
        ```
    - ```python
            spark_df = spark_df.withColumn("dob",F.expr("""
                                concat(day,'/',month,'/',year)
                            """))
      ```
        - This code will combine the vales in the column day, month and year but the problem is returned values will of string dataType
        - In order to fix it I will have to use ```to_date()``` function here in this case I will implement the ```to_date``` function inside the sql query 
        - ```python
            spark_df = spark_df.withColumn("dob",F.expr("""
                to_date(concat(day,'/',month,'/',year),'d/M/y')
            """))
          ```


## Spark aggregations
- Aggregations can be classified into three categories
    - Simple aggregations
    - Grouping aggregations
    - Windowing aggregations
- All aggregations are implemented in spark via built in functions.
- Here I am going to perform a simple aggregation. A simple aggregation will always give you a one line answer.
```python
def simple_aggregation_operation(self,spark_df):
        try:
            spark_df = spark_df.selectExpr(
                """
                    count(*) `count`
                """,
                """
                    count(StockCode) as `count field`
                """,
                """sum(Quantity) as TotalQuantity""",
                """avg(UnitPrice) as AverageUnitPrice"""
            )
            return spark_df
        except Exception as e:
            self.logger.error(str(e))
            raise
```
- But this is not enough I want more detailed information and get valuable insight.
    - ```python
            def group_by_country_agg(self,spark_df):
                try:
                    NumInvoice = F.countDistinct("InvoiceNo").alias("NumInvoice")
                    TotalQuantity = F.sum("Quantity").alias("TotalQuantity")
                    InvoiceValue = F.round(
                        F.sum(
                            F.expr("Quantity * UnitPrice")
                        ),2
                    ).alias("InvoiceValue")
                    saprk_df = (
                        spark_df
                        .where("year(InvoiceDate) == 2010")
                        .withColumn("WeekNumber",F.weekofyear(
                            F.col("InvoiceDate")
                        ))
                        .groupBy(
                            "Country","WeekNumber"
                        )
                        .agg(
                            NumInvoice, TotalQuantity, InvoiceValue
                        )
                    )
                    return saprk_df
                except Exception as e:
                    self.logger.error(str(e))
                    raise
      ```
- Performing window operations to perform aggregations
    - I want to compute running totals for each country. 
    - The running totals should re-start for each country
    - So the first thing is to make sure to break the dataFrame by the country
    - The next thing is to make sure is to order the rows in group by the WeekNumber, because I want to compute week by week number
    - The last thing to make sure to compute the running total by using a sliding window of records
### Error I faced:
- In the InvoiceDate column I was getting NAT (Not a time) in my dataFrame after ingesting data from the csv file where date-time is stored like this ```01-12-2010 8.26```
- Cause of this error : 
    ```python
    def return_invoice_df_schema(self):
            invoice_schema = StructType([
                StructField("InvoiceNo", IntegerType(), True),
                StructField("StockCode", StringType(), True),
                StructField("Description", StringType(), True),
                StructField("Quantity", IntegerType(), True),
                StructField("InvoiceDate", TimestampType(), True),
                StructField("UnitPrice", DoubleType(), True),
                StructField("CustomerID", IntegerType(), True),
                StructField("Country", StringType(), True)
            ])
            return invoice_schema
    ```
    - Normally when a date is in stringType where timestamp is present we can easily convert it to timestampType using the code below
    - The date in this column is like this ```01-12-2010 8.26```
    - Normally when we do this 
    ```python
        spark_df=spark_df = spark_df.withColumn(
                    "InvoiceDate",
                    F.to_timestamp(F.regexp_replace("InvoiceDate", r"\.", ":"), "dd-MM-yyyy H:mm")
                )
    ```
    - But since in the dataFrame schema I declared the Invoice date column as TimestampType I got NAT in this column after the ingestion of data from csv file was done.
- Solution :
    - Change the dataFrame schema for the InvoiceDate from TimestampType to StringType
    - ```python
        def return_invoice_df_schema(self):
            invoice_schema = StructType([
                StructField("InvoiceNo", IntegerType(), True),
                StructField("StockCode", StringType(), True),
                StructField("Description", StringType(), True),
                StructField("Quantity", IntegerType(), True),
                StructField("InvoiceDate", StringType(), True),
                StructField("UnitPrice", DoubleType(), True),
                StructField("CustomerID", IntegerType(), True),
                StructField("Country", StringType(), True)
            ])
            return invoice_schema
     ```

